# ⬛ OSINT Toolkit - Cloud Deployment Server
This notebook provisions a temporary, high-speed Cloudflare Tunnel to securely host your Streamlit dashboard. 

### Instructions:
1. Click the **Run** button (▶) on the code cell below.
2. Wait roughly 20-30 seconds for the environment to build.
3. Click the secure **`trycloudflare.com`** link generated at the bottom to access your live UI.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/kakarotoncloud/osint-toolkit.git
%cd osint-toolkit

# 2. Install Python dependencies quietly
!pip install -q -r requirements.txt

# 3. Download Cloudflare Tunnel daemon
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 4. Start Streamlit in the background
import subprocess
import time
import re

print("\n⚙️ Starting Streamlit engine...")
# Explicitly binding to 127.0.0.1 prevents Colab networking issues
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1", "--server.headless", "true"])

print("⏳ Waiting for Streamlit to boot up fully...")
time.sleep(7) # Extended buffer to prevent 502 Bad Gateway errors

# 5. Start Cloudflare Tunnel and parse the URL
print("🛡️ Initializing secure Cloudflare Tunnel...")
process = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://127.0.0.1:8501'], 
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("\n===================================================================")
for line in process.stdout:
    if ".trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n✅ YOUR DASHBOARD IS LIVE AT:")
            print(f"👉 {match.group(0)}\n")
            print("===================================================================")
            break